# Figure 1

This notebook produce Figure 1 in [Ronchi et al. 2026](...)

In [ ]:
import numpy as np
import os
import pandas as pd
import pathlib
import pickle
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.magneto_rotational_physics.magnetic_field_evolution as mre
import utilities.plot_settings

Import the magneto-thermal models that includes magnetic field decay, thermal luminosity evolution and crust failure events for five different values of the initial magnetic field: $10^{12}$ G, $10^{13}$ G, $10^{14}$ G, $10^{15}$ G and $5 \times 10^{15}$ G.

In [ ]:
base_path = pathlib.Path("../../")

if cfg["magneto-thermal_model"] == "BSk24_dip-tor_heavy":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e12_H.csv")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e13_H.csv")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e14_H.csv")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e15_H.csv")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_5e15_H.csv") 

    simB12_failure_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e12_H.d"
    )
    simB13_failure_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e13_H.d"
    )
    simB14_failure_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e14_H.d"
    )
    simB15_failure_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e15_H.d"
    )
    simB5e15_failure_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_5e15_H.d"
    )

    # Define file paths in a dictionary.
    file_paths_failures = {
        "1e12": simB12_failure_path,
        "1e13": simB13_failure_path,
        "1e14": simB14_failure_path,
        "1e15": simB15_failure_path,
        "5e15": simB5e15_failure_path,
    }

    # Define an array with the log10 of the initial magnetic field values for the different failure event sets.
    log_B0 = np.array([12, 13, 14, 15, np.log10(5.0e15)])

else:
    raise ValueError(
            "The magneto-thermal model provided in the configuration file is not correct to reproduce the results, choose BSk24_dip-tor_heavy."
        )

In [ ]:
df_simB12 = pd.read_csv(
    simB12_path,
    delimiter=",",
    header=[0],
)
df_simB12.head()

In [ ]:
df_simB13 = pd.read_csv(
    simB13_path,
    delimiter=",",
    header=[0],
)
df_simB13.head()

In [ ]:
df_simB14 = pd.read_csv(
    simB14_path,
    delimiter=",",
    header=[0],
)
df_simB14.head()

In [ ]:
df_simB15 = pd.read_csv(
    simB15_path,
    delimiter=",",
    header=[0],
)
df_simB15.head()

In [ ]:
df_simB5e15 = pd.read_csv(
    simB5e15_path,
    delimiter=",",
    header=[0],
)
df_simB5e15.head()

Save the time coordinate and the corresponding magnetic field and thermal luminosity evolutions in numpy arrays.

In [ ]:
t12_sim = df_simB12["t[yr]"].to_numpy().astype(np.float64)
t13_sim = df_simB13["t[yr]"].to_numpy().astype(np.float64)
t14_sim = df_simB14["t[yr]"].to_numpy().astype(np.float64)
t15_sim = df_simB15["t[yr]"].to_numpy().astype(np.float64)
t5e15_sim = df_simB5e15["t[yr]"].to_numpy().astype(np.float64)

B12_sim = df_simB12["B[G]"].to_numpy().astype(np.float64)
B13_sim = df_simB13["B[G]"].to_numpy().astype(np.float64)
B14_sim = df_simB14["B[G]"].to_numpy().astype(np.float64)
B15_sim = df_simB15["B[G]"].to_numpy().astype(np.float64)
B5e15_sim = df_simB5e15["B[G]"].to_numpy().astype(np.float64)

L12_sim = df_simB12["L[erg/s]"].to_numpy().astype(float)
L13_sim = df_simB13["L[erg/s]"].to_numpy().astype(float)
L14_sim = df_simB14["L[erg/s]"].to_numpy().astype(float)
L15_sim = df_simB15["L[erg/s]"].to_numpy().astype(float)
L5e15_sim = df_simB5e15["L[erg/s]"].to_numpy().astype(float)

Save the failures event information in a dataframe for easy accessibility.

In [ ]:
# Define names of columns with relevant information.
columns = ["time", "energy", "theta", "radius", "volume", "timestep"]

# Load DataFrames safely into a dictionary.
dfs_failures = {}
for key, path in file_paths_failures.items():
    # Check if the files are empty, i.e., contain no failure events.
    if os.path.getsize(path) > 0:
        df = pd.read_csv(path, sep=r"\s+", header=None)
        df.columns = columns
    else:
        df = pd.DataFrame(columns=columns)  # Assign columns even if empty.
    dfs_failures[key] = df

In [ ]:
t = {}

for key, df in dfs_failures.items():
    t[key] = df["time"].values

Compute the rate of crust failures using logarithmic bins.

In [ ]:
# Define bin edges.
t_edges = np.logspace(0.0, 7, 20)
bin_widths = np.diff(t_edges)
#bin_centers = t_edges[:-1] + bin_widths / 2
bin_centers = t_edges[:-1] + 10**(np.diff(np.log10(t_edges)) / 2)

keys = dfs_failures.keys()

rates = {}   # store results if needed later

for i, key in enumerate(keys):
    # Compute histogram.
    counts, _ = np.histogram(t[key], bins=t_edges)

    # Convert to rate
    rate = counts / bin_widths
    rates[key] = rate

Compute the magnetic field decay curves through the analytical fit (see also the notebook `tutorials/analysis_notebooks/magnetic_field_evolution_fit.ipynb`) and the thermal luminosity and failure rate evolution curves through the interpolator functions (see also the notebooks `xray_luminosity_interpolation.ipynb` and `crust_failure_rate_interpolation.ipynb` in `tutorials/analysis_notebooks`).

In [ ]:
# Define initial magnetic fields at which we evaluate the magnetic field decay fit, the interpolated thermal luminosity and failure rate curves.
# Note that this is needed now to set the right colors.
log_B0_eval = np.linspace(11.0, 16.0, 50)

# Combine the arrays to find the global min and max values.
combined_values = np.concatenate([log_B0, log_B0_eval])
vmin, vmax = combined_values.min(), combined_values.max()

# Create a colormap and normalize it.
cmap = plt.cm.viridis
norm = Normalize(vmin=vmin, vmax=vmax)

In [ ]:
log_B_initial = combined_values
B_initial = 10**log_B_initial
time = np.logspace(0.0, 8.0, 100)

In [ ]:
a1 = cfg["a1"]
a2 = cfg["a2"]
A1 = cfg["A1"]
A2 = cfg["A2"]
b1 = cfg["b1"]
b2 = cfg["b2"]
tau_late = cfg["tau_late"]
    
a_late = cfg["a_late"]

B_asymptotic = 10 ** np.random.normal(
    cfg["B_millisec_mean"], cfg["B_millisec_sigma"], len(B_initial)
)

B_fit = np.zeros((len(B_initial), len(time)))

for i in range(len(B_initial)):
    B_fit[i, :] = mre.magnetic_field_evolution_fit_numpy(
        B_initial[i], time, B_asymptotic[i], a1, a2, A1, A2, b1, b2, tau_late, a_late
    )

In [ ]:
# Load the interpolator functions for the X-ray luminosity and failures.
Lx_interpolator_path = base_path.joinpath(cfg["magneto-thermal_path"], "interpolator_Lx.pkl")

with open(
    Lx_interpolator_path,
    "rb",
) as f:
    Lx_interpolator_import = pickle.load(f)

failures_interpolator_path = base_path.joinpath(
    cfg["magneto-thermal_path"], "interpolator_crust_failure_rate.pkl"
)

with open(
    failures_interpolator_path,
    "rb",
) as f:
    failure_rate_interpolator_import = pickle.load(f)

In [ ]:
# Define a grid of times and initial magnetic fields at which we evaluate the interpolated cooling curves.
t_eval = np.logspace(0.0, 8, 500)
B0_eval = 10**log_B0_eval

Lt_interp = Lx_interpolator_import(t_eval, B0_eval)
failure_rate_interp = failure_rate_interpolator_import(t_eval, B0_eval)

Create the plot.

In [ ]:
# Create figure and subplots (shared x-axis)
fig, axes = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(12, 20), constrained_layout=True)

# Plot data
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[2].set_xscale("log")
axes[2].set_yscale("log")

axes[1].set_xlim(1.0, 1.0e8)
axes[2].set_xlim(1.0, 1.0e8)
axes[1].set_ylim(1.0e27, 1.0e37)
axes[2].set_ylim(1.0e-5, 2.0e3)


axes[0].plot(
    t12_sim,
    B12_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
axes[0].plot(
    t13_sim,
    B13_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
axes[0].plot(
    t14_sim,
    B14_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
axes[0].plot(
    t15_sim,
    B15_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
)
axes[0].plot(
    t5e15_sim,
    B5e15_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
)

for i in range(len(B_initial)):
    axes[0].plot(
        time,
        B_fit[i, :],
        linestyle="--",
        linewidth=4,
        color=cmap(norm(log_B_initial[i])),
        rasterized=True,
        alpha=0.5,
    )

axes[1].plot(
    t12_sim,
    L12_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
axes[1].plot(
    t13_sim,
    L13_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
axes[1].plot(
    t14_sim,
    L14_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
axes[1].plot(
    t15_sim,
    L15_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
)
axes[1].plot(
    t5e15_sim,
    L5e15_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
)

for i in range(len(B0_eval)):
    axes[1].plot(
        t_eval,
        Lt_interp[:, i],
        linestyle="--",
        linewidth=4,
        color=cmap(norm(log_B0_eval[i])),
        rasterized=True,
        alpha=0.5,
    )

for i, key in enumerate(keys):
    axes[2].plot(
        bin_centers,
        rates[key],
        lw=4,
        alpha=1,
        color=cmap(norm(log_B0[i])),
        rasterized=True,
    )

for i in range(len(B0_eval)):
    axes[2].plot(
        t_eval,
        failure_rate_interp[:, i],
        linestyle="--",
        linewidth=4,
        color=cmap(norm(log_B0_eval[i])),
        rasterized=True,
        alpha=0.5,
    )


# Remove x tick labels for middle and top plots
for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)

# Optional: remove spacing between subplots
plt.subplots_adjust(hspace=0)

# Create a shared colorbar
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(
    sm, 
    ax=axes, 
    #fraction=0.05, 
    #pad=0.02,
    shrink=0.4
)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

# Labels (only bottom subplot gets x-label)
axes[-1].set_xlabel(r"Time $t$ [yr]")
axes[0].set_ylabel(r"Magnetic field $B$ [G]")
axes[1].set_ylabel(r"Thermal luminosity $L_{\rm th}$ [erg s$^{-1}$]")
axes[2].set_ylabel("Failures per year")

axes[0].grid()
axes[1].grid()
axes[2].grid()
plt.show()

fig.savefig("plots/B_Lx_failure_evol.pdf")